# Notebook 10 – Feature Importance

## What is Feature Importance?

Feature importance measures how useful each feature is for a Machine Learning model when making predictions.

A feature with higher importance contributes more to the model's predictions than a feature with lower importance.

Feature importance helps us:
- Understand which features the model relies on.
- Identify useful features.
- Remove less useful features.
- Improve model interpretability.
- Support feature selection decisions.

### Important Note

A feature having high importance does **not automatically mean that it causes the target**.

Feature importance shows an association or usefulness to the model's predictions. It does not prove a causal relationship.

For example, if `Pclass` has high importance for predicting `Survived`, it means the model found `Pclass` useful for prediction. It does not mean that Pclass directly caused a passenger to survive.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score

df = pd.read_csv("Titanic-Dataset.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Model-Based Feature Importance

Model-based feature importance is calculated using a Machine Learning model.

Tree-based models such as Decision Trees and Random Forests can measure feature importance based on how much a feature helps split the data and improve the model.

Common model-based approaches include:
- Decision Tree Feature Importance
- Random Forest Feature Importance

The importance values can be compared to understand which features are more useful to the model.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]

X = df[features].fillna(df[features].median())
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

importance

,Feature,Importance
4,Fare,0.441256
1,Age,0.359246
0,Pclass,0.086332
2,SibSp,0.060757
3,Parch,0.052408


## 3. Decision Tree Feature Importance

A Decision Tree makes decisions by splitting the data using different features.

Feature importance shows how much each feature contributed to these splits.

A feature with a higher importance value had a greater contribution to the Decision Tree's predictions.

### Advantages
- Simple to understand.
- Easy to calculate.
- Useful for identifying important features.

### Limitation
The importance can be affected by correlated features and the structure of the tree.

In [3]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train, y_train)

tree_importance = pd.DataFrame({
    "Feature": features,
    "Importance": tree.feature_importances_
}).sort_values("Importance", ascending=False)

tree_importance

,Feature,Importance
4,Fare,0.429978
1,Age,0.259740
2,SibSp,0.161525
0,Pclass,0.098593
3,Parch,0.050165


In [4]:
rf_importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

rf_importance

,Feature,Importance
4,Fare,0.441256
1,Age,0.359246
0,Pclass,0.086332
2,SibSp,0.060757
3,Parch,0.052408


## 5. Permutation Importance

Permutation Importance measures how much the model performance changes when the values of one feature are randomly shuffled.

If shuffling a feature causes a large decrease in model performance, that feature is considered important.

It is useful because it measures the importance of a feature based on the model's actual prediction performance.

In [5]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    rf, X_test, y_test, n_repeats=5, random_state=42
)

permutation = pd.DataFrame({
    "Feature": features,
    "Importance": result.importances_mean
}).sort_values("Importance", ascending=False)

permutation

,Feature,Importance
4,Fare,0.018994
1,Age,0.017877
3,Parch,0.003352
2,SibSp,-0.007821
0,Pclass,-0.012291


## 6. Feature Importance vs Feature Selection

Feature Importance tells us how useful a feature is to a Machine Learning model.

Feature Selection is the process of choosing useful features and removing irrelevant or redundant features.

In simple terms:

- Feature Importance → Measures feature usefulness.
- Feature Selection → Decides which features to keep.

Feature importance can therefore be used as one input when performing feature selection.

In [6]:
importance.sort_values("Importance", ascending=False)

,Feature,Importance
4,Fare,0.441256
1,Age,0.359246
0,Pclass,0.086332
2,SibSp,0.060757
3,Parch,0.052408


## 7. Interpreting Feature Importance

A high importance value means that the feature was useful for the model's predictions.

However, importance values can change depending on the model, dataset, correlated features, and preprocessing.

Therefore, we should consider model performance, domain knowledge, correlation, and leakage before deciding whether to keep or remove a feature.

In [7]:
rf_importance.sort_values("Importance", ascending=False)

,Feature,Importance
4,Fare,0.441256
1,Age,0.359246
0,Pclass,0.086332
2,SibSp,0.060757
3,Parch,0.052408


## 8. Why Does High Feature Importance Not Mean Causation?

A feature with high importance does not automatically mean that it causes the target.

It only means that the model found the feature useful for making predictions.

For example, if Pclass has high importance when predicting Survived, it means Pclass provides useful information to the model. It does not prove that Pclass directly caused a passenger to survive.

Therefore:

**Feature Importance shows predictive usefulness, not causation.**

In [8]:
rf_importance.head()

,Feature,Importance
4,Fare,0.441256
1,Age,0.359246
0,Pclass,0.086332
2,SibSp,0.060757
3,Parch,0.052408


## 9. Limitations of Feature Importance

Feature Importance has some limitations:

- Different models can produce different importance values.
- Correlated features can affect importance values.
- Importance does not prove causation.
- Importance can change with different datasets.
- A low-importance feature may still be useful when combined with another feature.
- Data leakage can produce misleadingly high importance.

Therefore, feature importance should not be used blindly to remove features.

In [9]:
importance.describe()

,Importance
count,5.000000
mean,0.200000
std,0.185510
min,0.052408
25%,0.060757
50%,0.086332
75%,0.359246
max,0.441256


## 10. Required Documentation for Important Engineered Features

For every important engineered feature, document the following:

### Feature Name
Name of the newly created feature.

### Source Columns
Which columns were used to create the feature?

### Logic
Explain the formula or business rule used.

### Reason
Why was the feature created?

### Business Meaning
What does the feature represent?

### ML Relevance
How could the feature help a Machine Learning model?

### Leakage Check
Could this feature introduce data leakage?

### Final Decision
Choose one:
- Retain
- Remove
- Needs Further Analysis

In [10]:
documentation = pd.DataFrame([
    ["FamilySize", "SibSp, Parch", "SibSp + Parch + 1",
     "Represent family size", "Total family members",
     "May capture family survival patterns", "No target used", "Retain"],

    ["IsAlone", "FamilySize", "FamilySize == 1",
     "Identify passengers travelling alone", "Travelling alone or not",
     "May capture survival differences", "No target used", "Retain"],

    ["FarePerPerson", "Fare, FamilySize", "Fare / FamilySize",
     "Calculate fare per family member", "Approximate individual fare",
     "May provide useful fare information", "No target used",
     "Needs Further Analysis"],

    ["Age_Pclass_Interaction", "Age, Pclass", "Age * Pclass",
     "Capture age and class relationship", "Combined age-class information",
     "May capture feature interaction", "No target used", "Retain"],

    ["Fare_Log", "Fare", "log1p(Fare)",
     "Reduce fare skewness", "Log-scaled fare",
     "May improve model handling of skewed data", "No target used",
     "Needs Further Analysis"]
], columns=[
    "Feature Name",
    "Source Columns",
    "Logic",
    "Reason",
    "Business Meaning",
    "ML Relevance",
    "Leakage Check",
    "Final Decision"
])

documentation

,Feature Name,Source Columns,Logic,Reason,Business Meaning,ML Relevance,Leakage Check,Final Decision
0,FamilySize,"SibSp, Parch",SibSp + Parch + 1,Represent family size,Total family members,May capture family survival patterns,No target used,Retain
1,IsAlone,FamilySize,FamilySize == 1,Identify passengers travelling alone,Travelling alone or not,May capture survival differences,No target used,Retain
2,FarePerPerson,"Fare, FamilySize",Fare / FamilySize,Calculate fare per family member,Approximate individual fare,May provide useful fare information,No target used,Needs Further Analysis
3,Age_Pclass_Interaction,"Age, Pclass",Age * Pclass,Capture age and class relationship,Combined age-class information,May capture feature interaction,No target used,Retain
4,Fare_Log,Fare,log1p(Fare),Reduce fare skewness,Log-scaled fare,May improve model handling of skewed data,No target used,Needs Further Analysis


## 11. Conclusion

Feature Importance helps us understand which features are useful to a Machine Learning model.

In this notebook, we used Decision Tree Feature Importance, Random Forest Feature Importance, and Permutation Importance.

The main takeaway is that feature importance shows predictive usefulness, but it does not prove causation.

Feature importance should be considered together with feature selection, domain knowledge, model performance, and leakage checks.

In [11]:
print("Feature importance analysis completed successfully.")

Feature importance analysis completed successfully.
